# C-max20-ES — exploratory Dev-only optimization

Run this only after the controlled factorial artifacts and conclusion have been frozen. This is a fresh Model-A-to-expanded-pool trajectory with maximum 20 epochs, minimum 8 epochs, Dev-loss patience 4, and min_delta 1e-4. It is not a factorial cell and never loads Test.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
RUNTIME = Path('/kaggle/input/visolexnorm-offline-runtime')  # update slug if needed
REPO = Path('/kaggle/working/VisolexNorm')
BUNDLE = RUNTIME/'source/visolexnorm.bundle'
assert BUNDLE.is_file() and (RUNTIME/'wheelhouse').is_dir(), 'Attach the unpacked offline runtime Dataset.'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git', 'clone', str(BUNDLE), str(REPO)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
manifest = json.loads((RUNTIME/'offline-runtime-manifest.json').read_text())
assert SOURCE_COMMIT == manifest['source_commit']
%cd {REPO}
sys.path.insert(0, str(REPO))
from visolexnorm.common.offline_runtime import install_runtime
OFFLINE_PACKAGES = Path('/kaggle/working/offline_packages')
install_runtime(RUNTIME, OFFLINE_PACKAGES)
os.environ.update({'PIP_NO_INDEX':'1', 'HF_HUB_OFFLINE':'1', 'TRANSFORMERS_OFFLINE':'1'})
import torch, transformers, datasets, sentencepiece, safetensors
assert torch.cuda.is_available(), 'Enable the RTX Pro 6000 GPU accelerator.'
assert sys.version_info[:2] == (3, 12), sys.version
print('Offline source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0), '| transformers:', transformers.__version__)

In [ ]:
MOUNT = Path('/kaggle/input/visolexnorm-controlled-input')  # attach as an unpacked Dev-only private Dataset
WORK = Path('/kaggle/working/c_max20_es')
DATA = MOUNT  # Never duplicate Model A/data from /kaggle/input into /kaggle/working.
assert not (MOUNT/'controlled_training_input.zip').is_file(), 'Attach the input Dataset as an unpacked directory, not a ZIP.'
required = [DATA/'checkpoints/model_a/config.json', DATA/'data/processed/vilexnorm_train.jsonl', DATA/'data/processed/vilexnorm_dev.jsonl', DATA/'data/processed/visolex_weak_labeled_expanded.jsonl']
assert not [str(path) for path in required if not path.is_file()]
assert not (DATA/'data/processed/vilexnorm_test.jsonl').exists()
assert not (DATA/'outputs/evaluation').exists()
WORK.mkdir(parents=True, exist_ok=True)
def show_disk(label):
    usage = shutil.disk_usage('/kaggle/working')
    print(f'{label}: free={usage.free / 2**30:.2f} GiB, used={usage.used / 2**30:.2f} GiB')
show_disk('Before C-max20')

In [ ]:
# Copy the frozen protocol downloaded from the completed factorial run to this Kaggle session before this cell.
PROTOCOL = Path('/kaggle/input/controlled-factorial-artifacts/controlled_factorial_protocol.json')  # update dataset/filename
assert PROTOCOL.is_file(), 'Attach the frozen factorial protocol artifact.'
RUN = WORK/'seed_2026'
!python -m scripts.controlled_experiments build-c-max20 --data-root {DATA} --config {REPO}/configs/c_max20_early_stopping_config.json --protocol {PROTOCOL} --seed 2026 --output {RUN}/mixture_manifest.json

In [ ]:
# Smoke test only; full run below starts freshly from Model A.
SMOKE = WORK/'smoke'
!python -m scripts.controlled_experiments train-c-max20 --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --manifest {RUN}/mixture_manifest.json --config {REPO}/configs/c_max20_early_stopping_config.json --work-dir {SMOKE} --smoke-test --no-resume-state
smoke = json.loads((SMOKE/'smoke_test.json').read_text())
assert smoke['passed'] and smoke['test_inputs_loaded'] is False, smoke
shutil.rmtree(SMOKE)
show_disk('After C-max20 smoke cleanup')

In [ ]:
# Disk-safe mode does not write a large AdamW resume state. If interrupted, delete RUN/state and RUN/best, then rerun this fresh trajectory from Model A.
!python -m scripts.controlled_experiments train-c-max20 --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --manifest {RUN}/mixture_manifest.json --config {REPO}/configs/c_max20_early_stopping_config.json --work-dir {RUN} --no-resume-state

In [ ]:
report = json.loads((RUN/'early_stopping_report.json').read_text())
assert report['rule']['min_epochs'] == 8 and report['rule']['patience'] == 4
assert report['test_inputs_loaded'] is False
assert not (RUN/'state').exists(), 'Disk-safe run should not write resume state.'
show_disk('Before C-max20 archive')
shutil.make_archive('/kaggle/working/c_max20_es_artifacts', 'zip', WORK, 'seed_2026')
print(report)
print('Download c_max20_es_artifacts.zip. This result is exploratory optimization, not a causal factorial result.')